# NegPairs v3.0 build: NegPairs-Perm and NegPairs-Fresh

**Runtime:** GPU **L4** (recommended, cheaper) or **A100**. High-RAM is **not** needed. A T4 does not work (Qwen needs
bfloat16 and about 16 GB of GPU memory); the notebook stops on it.
**Secrets:** `ANTHROPIC_API_KEY` in the Colab Secrets panel with *Notebook access* on (`HF_TOKEN` optional).
**Files:** `cacsa2_project.zip` (0.3.0) in `MyDrive/CACSA_v3/`. Then `Runtime > Run all`.

**What it does (manifest v3.0, frozen):**
1. Checks the frozen manifest, the prompt hashes and the NegPairs v1 files, and records the manifest hash **before any API call**.
2. Loads Qwen2.5-7B-Instruct (second verifier, another model family).
3. Phase 1, **NegPairs-Perm**: permutation minimal pairs (N and S contain the same words; the negation sits in the
   decisive clause in N and in the irrelevant clause in S). Cap 38 per family and clause order (304 in total).
4. Phase 2, **NegPairs-Fresh**: new test set with the v2.0 pipeline. Cap 60 per family (240 in total).
5. Opus audit of a seeded 10% sample of each set; report; zip.

Every candidate: generated by Claude Sonnet 5, filtered, verified arm by arm by Claude Haiku 4.5, then by Qwen (both
option orders); accepted in schedule order. Everything is saved to Drive after every batch: if the session stops,
run all cells again; nothing is paid twice.

| Step | Expected |
|---|---|
| setup, checks, Qwen download | 5 to 10 min |
| phase 1 (Perm) | about 1 to 1.5 h, one line per batch |
| phase 2 (Fresh) | about 45 to 75 min |
| audit, report, zip | 5 to 10 min |

**Budget (metered):** Perm stops at 12 USD, Fresh at 22 USD in total, the audit at 24 USD in total.

**Status of the data.** Model-generated and model-verified (two model families), **not human-verified**.

At the end, `negpairs_v3.zip` is downloaded (a copy stays in `MyDrive/CACSA_v3/`). Send this zip back.


In [ ]:
# ============================== CONFIGURATION (manifest v3.0) ==============================
PROJECT_ROOT  = "/content/drive/MyDrive/CACSA_v3"
PACKAGE_ZIP   = f"{PROJECT_ROOT}/cacsa2_project.zip"
OUT_NAME      = "negpairs_v3"
MANIFEST_FILE = "CACSA_manifest_v3_0.md"
MANIFEST_SHA256 = "57f834c56455cb7cd6b2c997f3599a86b2b7dbf68213dea8986ef2fd1fb1561c"
EXPECTED_PROMPT_SHA256 = {
 "PERM_GEN_PROMPT": "12ecbd2475c43497d160e8e2c4b94af2979e8ca7c13c2f8d4793287b531dfc74",
 "QWEN_VERIFY_PROMPT": "818f46ad91106d988ccc5349bfb891b98bc8a1400ff6ade153f022b62f60661d",
 "PERM_PATTERNS.syntactic.pattern_desc": "4b72ae115883be933e21c1e4f806a36502b0d1a359f5da57390cc9a9d46be3da",
 "PERM_PATTERNS.syntactic.negated_form_desc": "c1ec01050e8b58dd2e96f774d507b47a4d393614b4a24ceb7279541390fc4349",
 "PERM_PATTERNS.quantifier.pattern_desc": "c872365872da723ca30429af162aab7133e048bf89b4f35192c8af19fd89b9c9",
 "PERM_PATTERNS.quantifier.negated_form_desc": "dbf7116b6b84c88d375fdc981cd6b1cf17f7d4a1586025df4b681fa8ecfae46e",
 "PERM_PATTERNS.affixal.pattern_desc": "00c69444f7251a0dae28590a80bd47666c33df52b0dcc9e5e5a167abc5fa9f67",
 "PERM_PATTERNS.affixal.negated_form_desc": "677be83bc308a36b997ee08b5c470710930c3a335a036775c50136b17e99e282",
 "PERM_PATTERNS.lexical.pattern_desc": "8510dc14de0895cf073179b84b02322c1db5a735657447e968f91d0ea44a3f65",
 "PERM_PATTERNS.lexical.negated_form_desc": "a8e17098bea3776fdd914a569ff3a809fa02f642de0f1084467e6c34ce689ea6",
 "GEN_PROMPT": "8270c1684edaadff9941da5244a8812e7fcdc62b95ada62ea21bfa38e668af35",
 "VERIFY_PROMPT_V2": "1ce31ce47f63c8c7903b22ba4869b5a99c0262a2f9356fec52faa117ce8c3fdd"
}
PERM  = dict(per_family=600, cap=38, seed=20261020, order_seed=20261022, phase_cap_usd=12.00)
FRESH = dict(per_family=480, cap=60, seed=20261005, phase_cap_usd=22.00)
HARD_CAP_USD  = 24.00
GEN_MODEL, GEN_EFFORT, GEN_MAX_TOKENS = "claude-sonnet-5", "low", 4000
VERIFY_MODEL  = "claude-haiku-4-5-20251001"
AUDIT_MODEL, AUDIT_EFFORT, AUDIT_MAX_TOKENS, AUDIT_FRACTION = "claude-opus-5", "low", 2000, 0.10
QWEN_MODEL, QWEN_BATCH = "Qwen/Qwen2.5-7B-Instruct", 16
BATCH_SIZE, MAX_WORKERS = 24, 8
# ===========================================================================================


In [ ]:
%pip install -q anthropic wordfreq==3.1.1 accelerate
from google.colab import drive
drive.mount('/content/drive')
import os, sys, json, zipfile, time, random, hashlib, shutil, importlib, subprocess, csv
import importlib.metadata as md
from pathlib import Path
from collections import Counter
from datetime import datetime, timezone
ROOT = Path(PROJECT_ROOT)
assert Path(PACKAGE_ZIP).exists(), f'upload cacsa2_project.zip (0.3.0) to {PROJECT_ROOT}/'
ZIP_SHA = hashlib.sha256(open(PACKAGE_ZIP, 'rb').read()).hexdigest()
PKG_DIR = Path('/content') / f'cacsa2_pkg_{ZIP_SHA[:12]}'          # fresh folder per zip version (no stale files)
shutil.rmtree(PKG_DIR, ignore_errors=True)
with zipfile.ZipFile(PACKAGE_ZIP) as z:
    z.extractall(PKG_DIR)
for _m in [m for m in list(sys.modules) if m == 'cacsa2' or m.startswith('cacsa2.')]:
    del sys.modules[_m]
sys.path = [p for p in sys.path if 'cacsa2_pkg_' not in p]
PKG_ROOT = next(PKG_DIR.rglob('cacsa2/__init__.py')).parent.parent
sys.path.insert(0, str(PKG_ROOT))
importlib.invalidate_caches()
import cacsa2
assert tuple(int(x) for x in cacsa2.__version__.split('.')) >= (0, 3, 0), \
    f'cacsa2_project.zip on Drive is version {cacsa2.__version__}; upload the newest one (>= 0.3.0)'
from cacsa2 import negpairs_build as nb, perm_build as pb, perm_prompts as pp, qwen_verify as qv, build_v3 as bv, provenance
OUT = ROOT / 'data' / OUT_NAME
OUT.mkdir(parents=True, exist_ok=True)
STATE = OUT / 'build_state.json'
API_KEY = os.environ.get('ANTHROPIC_API_KEY', '')
if not API_KEY:
    try:
        from google.colab import userdata
        API_KEY = userdata.get('ANTHROPIC_API_KEY')
    except Exception:
        pass
assert API_KEY, 'No API key: Secrets panel -> add ANTHROPIC_API_KEY -> switch on Notebook access.'
GPU = (subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], capture_output=True,
                      text=True).stdout.strip() if shutil.which('nvidia-smi') else 'none')
print('cacsa2', cacsa2.__version__, '| zip sha256', ZIP_SHA[:12], '| GPU:', GPU, '| output folder', OUT)


In [ ]:
# ---- integrity checks BEFORE any API call: frozen manifest, prompt hashes, wordfreq, NegPairs v1 ------------------
MAN = PKG_ROOT / MANIFEST_FILE
_sha = provenance.sha256_file(MAN)
assert _sha == MANIFEST_SHA256, f'packaged manifest {_sha[:12]} differs from the frozen manifest {MANIFEST_SHA256[:12]}'
LOCK = OUT / 'manifest_lock.json'
if LOCK.exists():
    assert json.loads(LOCK.read_text())['sha256'] == _sha, 'manifest_lock.json in the output folder belongs to another manifest'
else:
    LOCK.write_text(json.dumps(provenance.lock_record(MAN, 'recorded before the first API call of the v3.0 build'), indent=1))
shutil.copy(MAN, OUT / MANIFEST_FILE)
_got = {**pp.prompt_hashes(), 'GEN_PROMPT': nb.prompt_hashes()['GEN_PROMPT'],
        'VERIFY_PROMPT_V2': nb.prompt_hashes()['VERIFY_PROMPT_V2']}
_bad = sorted(k for k, v in EXPECTED_PROMPT_SHA256.items() if _got.get(k) != v)
assert not _bad, f'prompt hashes differ from the manifest: {_bad}'
assert md.version('wordfreq') == '3.1.1', f"wordfreq {md.version('wordfreq')}; the manifest fixes 3.1.1"
V1 = PKG_ROOT / 'data' / 'negpairs_v1'
for row in csv.DictReader(open(V1 / 'FILE_MANIFEST_SHA256.csv', encoding='utf-8')):
    assert provenance.sha256_file(V1 / row['path']) == row['sha256'], f"NegPairs v1 file changed: {row['path']}"
V1_SCEN = [json.loads(l) for l in open(V1 / 'scenarios.jsonl', encoding='utf-8')]
assert len(V1_SCEN) == 1088, len(V1_SCEN)
print(f'manifest {_sha[:12]} locked | {len(_got)} prompt hashes match | wordfreq 3.1.1 | {len(V1_SCEN)} NegPairs v1 scenarios verified')


In [ ]:
# ---- Qwen2.5-7B-Instruct (cross-family verifier, manifest 4.4) ------------------------------------------------
try:
    from google.colab import userdata
    _hf = userdata.get('HF_TOKEN')
    if _hf:
        os.environ['HF_TOKEN'] = _hf
except Exception:
    pass
t0 = time.time()
QWEN = qv.QwenVerifier(QWEN_MODEL, batch_size=QWEN_BATCH)
print(f'Qwen loaded in {(time.time() - t0) / 60:.1f} min | revision {QWEN.revision} | letter token ids {QWEN.letter_ids}')
# sanity check on four NegPairs v1 train scenarios (printed only; no effect on any decision)
for s in [s for s in V1_SCEN if s['split'] == 'train'][:4]:
    _arms = {a: s['context_' + a] for a in 'OPNS'}
    _ok, _det = qv.scenario_check(QWEN.logits3(QWEN.scenario_texts(_arms, s['question'], s['options'])), s['answer_O'])
    print(' ', s['uid'], 'pass' if _ok else 'FAIL', {a: _det[a]['verdict'] for a in 'OPNS'}, 'expected',
          {a: _det[a]['expected'] for a in 'OPNS'})


In [ ]:
# ---- record on Drive, metered API calls ------------------------------------------------------------------------
state = json.loads(STATE.read_text()) if STATE.exists() else {}
for _k in ('perm', 'fresh'):
    state.setdefault(_k, {'done': {}})
state.setdefault('audit', {'perm': {}, 'fresh': {}})
state['qwen_revision'] = QWEN.revision
METER = bv.Meter(state.get('meter'))
def save_state():
    state['meter'] = METER.to_dict()
    tmp = STATE.with_suffix('.tmp')
    tmp.write_text(json.dumps(state, ensure_ascii=False))
    tmp.replace(STATE)
save_state()
from anthropic import Anthropic
CLIENT = Anthropic(api_key=API_KEY, max_retries=6, timeout=180.0)
call_gen = bv.make_call(CLIENT, METER, effort=GEN_EFFORT)             # generator
call_ver = bv.make_call(CLIENT, METER, temperature=0.0)               # Haiku verifier (deterministic)
call_audit = bv.make_call(CLIENT, METER, effort=AUDIT_EFFORT)         # Opus audit

PERM_SCHED = pb.perm_schedule(PERM['per_family'], PERM['seed'], PERM['order_seed'])
FRESH_SCHED = pb.fresh_schedule(FRESH['per_family'], FRESH['seed'])
perm_stratum_c = lambda c: f"{c.family}|{'first' if c.decisive_first else 'second'}"
perm_stratum_r = lambda r: f"{r['meta']['family']}|{'first' if r['meta']['decisive_first'] else 'second'}"
PERM_CAPS = {f'{f}|{o}': PERM['cap'] for f in nb.FAMILIES for o in ('first', 'second')}
FRESH_CAPS = {f: FRESH['cap'] for f in nb.FAMILIES}
perm_key = lambda r: (r['meta']['domain'], r['assembled']['O'], r['generated']['options'])
fresh_key = lambda r: (r['meta']['domain'], r['generated']['context_O'], r['generated']['options'])
DUP = nb.NearDuplicateIndex(threshold=0.6)                              # v1, then accepted Perm, then accepted Fresh
for s in V1_SCEN:
    DUP.add(s['domain'], s['context_O'], s['options'])
for _kind, _key in (('perm', perm_key), ('fresh', fresh_key)):
    for r in sorted((r for r in state[_kind]['done'].values() if r.get('stage') == 'accepted'), key=lambda r: r['meta']['index']):
        DUP.add(*_key(r))
is_dup = DUP.is_duplicate
FINAL = ('caps reached', 'schedule exhausted', 'budget')

def show(kind, n=1):
    acc = sorted((r for r in state[kind]['done'].values() if r.get('stage') == 'accepted'), key=lambda r: r['meta']['index'])
    for fam in nb.FAMILIES:
        for r in [r for r in acc if r['meta']['family'] == fam][:n]:
            arms, q, opts, ans = pb.scenario_arms(r, kind)
            print(f"\n[{kind} | {fam}] Q: {q}  A) {opts[0]}  B) {opts[1]}  answer {ans}")
            for a in 'OPNS':
                print(f'   {a}: {arms[a]}')

def report_phase(kind, caps, stratum_r):
    S = bv.summarize(state[kind]['done'], stratum_r, caps)
    print(json.dumps({k: S[k] for k in ('candidates', 'stages', 'accepted_by_stratum', 'qwen')}, indent=1))
    print('top discard reasons:', list(S['discard_reasons'].items())[:12])
    return S

print(f"record: Perm {len(state['perm']['done'])} | Fresh {len(state['fresh']['done'])} candidates | metered so far ${METER.usd:.2f}")


In [ ]:
# ---- phase 1: NegPairs-Perm (manifest 4.3) ----------------------------------------------------------------------
P = state['perm']
if P.get('halted') in FINAL:
    print('Perm phase already finished:', P['halted'])
else:
    METER.cap = PERM['phase_cap_usd']
    P['halted'] = bv.run_phase(
        PERM_SCHED, P['done'], stratum_c=perm_stratum_c, stratum_r=perm_stratum_r, caps=PERM_CAPS,
        api_task=lambda c: pb.process_perm(c, call_gen, call_ver, GEN_MODEL, VERIFY_MODEL, is_dup, GEN_MAX_TOKENS),
        settle_qwen=lambda recs: qv.verify_records(QWEN, recs, lambda r: pb.scenario_arms(r, 'perm')),
        final_key=perm_key, dup_index=DUP, save=save_state, log=lambda s: print(s, flush=True), label='perm',
        meter=METER, batch_size=BATCH_SIZE, max_workers=MAX_WORKERS)
    save_state()
print('\nPerm phase:', P['halted'], f'| metered ${METER.usd:.2f}')
PERM_SUMMARY = report_phase('perm', PERM_CAPS, perm_stratum_r)
show('perm')
if P['halted'] not in FINAL:
    raise RuntimeError(f"Perm phase stopped early ({P['halted']}). Fix the cause (for example API credit) and run all "
                       f"cells again; nothing is paid twice. Fresh starts only after Perm has finished.")


In [ ]:
# ---- phase 2: NegPairs-Fresh (manifest 4.2; v2.0 pipeline unchanged + Qwen) ----------------------------------------
def fresh_task(c):
    r = nb.process_v2(c, call_gen, call_ver, GEN_MODEL, VERIFY_MODEL, is_dup, GEN_MAX_TOKENS)
    r['meta'] = {**r.get('meta', {}), 'index': c.index}
    if r.get('stage') == 'kept':
        r['stage'] = 'haiku_passed'
    return r
F = state['fresh']
if F.get('halted') in FINAL:
    print('Fresh phase already finished:', F['halted'])
else:
    METER.cap = FRESH['phase_cap_usd']
    F['halted'] = bv.run_phase(
        FRESH_SCHED, F['done'], stratum_c=lambda c: c.family, stratum_r=lambda r: r['meta']['family'], caps=FRESH_CAPS,
        api_task=fresh_task,
        settle_qwen=lambda recs: qv.verify_records(QWEN, recs, lambda r: pb.scenario_arms(r, 'fresh')),
        final_key=fresh_key, dup_index=DUP, save=save_state, log=lambda s: print(s, flush=True), label='fresh',
        meter=METER, batch_size=BATCH_SIZE, max_workers=MAX_WORKERS)
    save_state()
print('\nFresh phase:', F['halted'], f'| metered ${METER.usd:.2f}')
FRESH_SUMMARY = report_phase('fresh', FRESH_CAPS, lambda r: r['meta']['family'])
show('fresh')
if F['halted'] not in FINAL:
    raise RuntimeError(f"Fresh phase stopped early ({F['halted']}). Fix the cause and run all cells again; nothing is paid twice.")


In [ ]:
# ---- Opus audit of a seeded 10% sample of each set (manifest 4.5; descriptive) ------------------------------------
METER.cap = HARD_CAP_USD
AUDIT = {}
for kind, conf in (('perm', PERM), ('fresh', FRESH)):
    acc = [r for r in state[kind]['done'].values() if r.get('stage') == 'accepted']
    sample = bv.audit_sample([r['meta']['uid'] for r in acc], conf['seed'], AUDIT_FRACTION)
    gv_of = (lambda r: pb.perm_as_v2(r['generated'], r['assembled'])) if kind == 'perm' else (lambda r: r['generated'])
    AUDIT[kind] = bv.run_audit(acc, gv_of, call_audit, AUDIT_MODEL, sample, AUDIT_MAX_TOKENS, state['audit'][kind],
                               save_state, print)
    print(f"{kind} audit: {AUDIT[kind]['agree_all_arms']}/{AUDIT[kind]['checked']} sampled scenarios agree on all four "
          f"arms (sample {AUDIT[kind]['sample']}{', stopped: ' + AUDIT[kind]['stopped'] if AUDIT[kind]['stopped'] else ''})")
save_state()
print(f'metered total ${METER.usd:.2f}')


In [ ]:
# ---- item files, build-time assertion, report, file manifest, zip ---------------------------------------------------
ITEMS = {}
for kind, conf in (('perm', PERM), ('fresh', FRESH)):
    acc = [r for r in state[kind]['done'].values() if r.get('stage') == 'accepted']
    ITEMS[kind] = pb.to_items_v3(acc, kind, kind, conf['seed'])
    with open(OUT / f'{kind}.jsonl', 'w', encoding='utf-8') as fh:
        for it in ITEMS[kind]:
            fh.write(json.dumps(it, ensure_ascii=False) + '\n')
    with open(OUT / f'{kind}_scenarios.jsonl', 'w', encoding='utf-8') as fh:
        for r in sorted(acc, key=lambda r: r['meta']['uid']):
            fh.write(json.dumps(r, ensure_ascii=False) + '\n')
_ctx = {}
for it in ITEMS['perm']:
    _ctx.setdefault(it['scenario_id'], {})[it['arm']] = it['context']
BAG_OK = sum(pb.e0_bag(v['N']) == pb.e0_bag(v['S']) and v['N'] != v['S'] for v in _ctx.values())
assert BAG_OK == len(_ctx), 'a Perm scenario has unequal E0 bags in N and S'
def counts(kind):
    out = Counter((it['family'], (it['perm']['decisive_first'] if kind == 'perm' else None)) for it in ITEMS[kind] if it['arm'] == 'O')
    return {f"{f}" + ('' if o is None else ('|first' if o else '|second')): n for (f, o), n in sorted(out.items(), key=str)}
report = {
    'built_utc': datetime.now(timezone.utc).isoformat(), 'cacsa2_version': cacsa2.__version__,
    'package_zip_sha256': ZIP_SHA, 'manifest': {'file': MANIFEST_FILE, 'sha256': MANIFEST_SHA256},
    'models': {'generator': GEN_MODEL, 'generator_effort': GEN_EFFORT, 'generator_max_tokens': GEN_MAX_TOKENS,
               'verifier': VERIFY_MODEL, 'verifier_temperature': 0.0, 'audit': AUDIT_MODEL, 'audit_effort': AUDIT_EFFORT,
               'qwen': QWEN_MODEL, 'qwen_revision': QWEN.revision, 'qwen_dtype': 'bfloat16 weights, float32 letter logits'},
    'prompt_sha256': _got, 'price_table_usd_per_million_tokens': bv.PRICES,
    'seeds': {'perm_schedule': PERM['seed'], 'perm_clause_order': PERM['order_seed'], 'perm_option_order': PERM['seed'] + 1,
              'perm_audit': PERM['seed'] + 7, 'fresh_schedule': FRESH['seed'], 'fresh_option_order': FRESH['seed'] + 1,
              'fresh_audit': FRESH['seed'] + 7},
    'scheduled_per_family': {'perm': PERM['per_family'], 'fresh': FRESH['per_family']},
    'budget_caps_usd': {'perm_phase': PERM['phase_cap_usd'], 'fresh_phase': FRESH['phase_cap_usd'], 'hard': HARD_CAP_USD},
    'perm': {**bv.summarize(state['perm']['done'], perm_stratum_r, PERM_CAPS), 'halted': state['perm']['halted'],
             'accepted_by_family_and_order': counts('perm'),
             'bag_assertion': {'scenarios': len(_ctx), 'equal_E0_bags_N_S': BAG_OK},
             'label_balance': {a: dict(Counter(it['label'] for it in ITEMS['perm'] if it['arm'] == a)) for a in 'OPNS'}},
    'fresh': {**bv.summarize(state['fresh']['done'], lambda r: r['meta']['family'], FRESH_CAPS),
              'halted': state['fresh']['halted'], 'accepted_by_family': counts('fresh'),
              'label_balance': {a: dict(Counter(it['label'] for it in ITEMS['fresh'] if it['arm'] == a)) for a in 'OPNS'}},
    'opus_audit': AUDIT, 'cost_metered': METER.to_dict(), 'gpu': GPU,
    'environment': provenance.run_environment(),
    'human_verification': 'not performed (author decision, 5 October 2026)',
}
(OUT / 'build_report.json').write_text(json.dumps(report, indent=1, ensure_ascii=False))
(OUT / 'README.md').write_text(f'''# NegPairs v3.0 sets (manifest v3.0, section 4)

Built {report['built_utc'][:10]} with cacsa2 {cacsa2.__version__} under the frozen manifest (SHA-256 {MANIFEST_SHA256}).

perm.jsonl: NegPairs-Perm, {len(_ctx)} scenarios, one row per arm (O, P, N, S). N and S contain the same words; the
negation edit sits in the decisive clause in N and in the other clause in S. Field `perm` gives the pattern, the clause
order and character spans (clauses, cue, unedited W) of each arm.
fresh.jsonl: NegPairs-Fresh, {len(ITEMS['fresh']) // 4} scenarios, built with the v2.0 pipeline.
*_scenarios.jsonl: accepted records with the generator output, Haiku verdicts and Qwen logits.
build_state.json: every candidate and the reason it was discarded. opus_audit in build_report.json.

Generated by {GEN_MODEL}; verified arm by arm by {VERIFY_MODEL} and by {QWEN_MODEL} (both option orders); a seeded 10%
sample audited by {AUDIT_MODEL}. Not human-verified. Metered cost ${METER.usd:.2f}.
''')
n = provenance.write_file_manifest(OUT, OUT / 'FILE_MANIFEST_SHA256.csv')
ZIP = Path('/content') / f'{OUT_NAME}.zip'
with zipfile.ZipFile(ZIP, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in sorted(x for x in OUT.rglob('*') if x.is_file() and x.suffix != '.tmp'):
        z.write(p, arcname=str(p.relative_to(OUT.parent)))
shutil.copy(ZIP, ROOT / f'{OUT_NAME}.zip')
print(json.dumps({'perm': report['perm']['accepted_by_family_and_order'], 'fresh': report['fresh']['accepted_by_family'],
                  'audit': {k: (v['agree_all_arms'], v['checked']) for k, v in AUDIT.items()},
                  'metered_usd': round(METER.usd, 2)}, indent=1))
print(f'{n} files manifested -> {ZIP} ({ZIP.stat().st_size // 1024} kB); a copy is in {ROOT}')
from google.colab import files
files.download(str(ZIP))
